In [27]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data import create_dummy_patient_data
from src.experiment import run_tree_experiment
from src.evaluation import plot_confusion_matrix, show_classification_report

In [28]:
data_path = PROJECT_ROOT / "data" / "titanic" / "train.csv"
train_df = pd.read_csv(data_path)
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [29]:
# Repeat for the test data
test_data_path = PROJECT_ROOT / "data" / "titanic" / "test.csv"
test_df = pd.read_csv(test_data_path)

In [30]:
# df, feature_names, classes = create_dummy_patient_data()
# df.head()
def feature_names_and_classes(df):
    feature_names = df.columns.drop("Survived").tolist()  # Create a list of feature names by dropping the target column
    classes = df["Survived"].unique().tolist()  # Identify the unique classes in the target column
    return feature_names, classes

def keep_relevant_features(df, irrelevant_features=None):
    # Keep only relevant features for the Titanic dataset
    return df.drop(columns=irrelevant_features)



# Keep only relevant features for the Titanic dataset
train_df=keep_relevant_features(train_df, irrelevant_features=(["Name","Ticket","Cabin"]))
test_df=keep_relevant_features(test_df, irrelevant_features=(["Name","Ticket","Cabin"]))


# Section to Handle Missing Values

In [32]:
check_missing_values = titanic_df.isnull().sum()
print("Missing values in each column:\n", check_missing_values)

Missing values in each column:
 PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         2
dtype: int64


In [33]:
def fill_missing_values(df):
    # Fill missing values in the "Age" column with the median age
    df["Age"] = df["Age"].fillna(df["Age"].median())

    # Fill missing values in the "Embarked" column with the most common embarkation point
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # Convert categorical columns to numerical using one-hot encoding
    df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
    return df

train_df = fill_missing_values(train_df)
test_df = fill_missing_values(test_df)

In [35]:
train_df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,22.0,1,0,7.2500,True,False,True
1,2,1,1,38.0,1,0,71.2833,False,False,False
2,3,1,3,26.0,0,0,7.9250,False,False,True
3,4,1,1,35.0,1,0,53.1000,False,False,True
4,5,0,3,35.0,0,0,8.0500,True,False,True


In [34]:
results = run_tree_experiment(
    df=train_df,
    target_col="Survived",
    feature_names=feature_names,
    results_dir="results/titanic"
)

ValueError: All arrays must be of the same length

In [ ]:
dt_results = results["Decision Tree"]

print("Decision Tree Accuracy:", dt_results["accuracy"])

plot_confusion_matrix(
    dt_results["y_test"],
    dt_results["predictions"],
    classes
)

show_classification_report(
    dt_results["y_test"],
    dt_results["predictions"]
)

In [ ]:
rf_results = results["Random Forest"]

print("Random Forest Accuracy:", rf_results["accuracy"])

plot_confusion_matrix(
    rf_results["y_test"],
    rf_results["predictions"],
    classes
)

show_classification_report(
    rf_results["y_test"],
    rf_results["predictions"]
)

In [ ]:
rf_results["feature_importance"]